In [1]:
import urllib
from transformers import pipeline
import os
import json
from datetime import datetime
import gradio as gr
import logging
from sentence_transformers import SentenceTransformer, util
from gliner import GLiNER
import xml.etree.ElementTree as ET
from dotenv import load_dotenv

load_dotenv()

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


True

In [2]:
os.makedirs("logs", exist_ok=True)
logging.basicConfig(filename="logs/app.log", level=logging.INFO, format="%(message)s")
transcribe_logger = logging.getLogger("transcribe")
transcribe_logger.propagate = False
transcribe_logger.setLevel(logging.INFO)
if not transcribe_logger.handlers:
    handler = logging.FileHandler("logs/transcribe.log")
    handler.setFormatter(logging.Formatter("%(message)s"))
    transcribe_logger.addHandler(handler)

In [3]:
asr = pipeline(task="automatic-speech-recognition",
               model="distil-whisper/distil-small.en",
               model_kwargs={"cache_dir": "models"})

print('asr_freq:', asr.feature_extractor.sampling_rate)

Loading weights: 100%|██████████| 287/287 [00:00<00:00, 15792.06it/s]


asr_freq: 16000


In [4]:
intent_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2",
                                    device="cuda",
                                    model_kwargs={"cache_dir": "models"})

intents = ["news", "ecommerce"]
intent_descriptions = ["latest news or headlines about a person or topic",
                       "shop for a product on Amazon, eBay, or Flipkart"]
intent_emb = intent_model.encode(intent_descriptions, normalize_embeddings=True)

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.79it/s]


In [5]:
ner_model = GLiNER.from_pretrained("urchade/gliner_small-v2.1",
                                    map_location="cuda",
                                    cache_dir="models")

/home/rakesh/Downloads/ShortTok/1.VoiceModel/VoiceSearch/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:190: UserWarning: The `resume_download` argument is deprecated and ignored in `snapshot_download`. Downloads always resume whenever possible.
  warnings.warn(
Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 1355.73it/s]


In [6]:
def inference_hardware():
    on_gpu = asr.device.type == "cuda"
    in_colab = "COLAB_RELEASE_TAG" in os.environ
    if on_gpu and in_colab:
        return "colab gpu"
    if on_gpu:
        return "local gpu"
    return "cpu"

def log_transcription(filepath, text, latency_ms):
    if filepath is not None:
        audio_dir = os.path.abspath("logs/audio")
        src = os.path.abspath(filepath)
        if src.startswith(audio_dir + os.sep):
            stem = "".join(c for c in text.strip() if c.isalnum() or c in " ,.'-")[:80].strip() or "audio"
            ext = os.path.splitext(src)[1]
            dest = os.path.join(audio_dir, stem + ext)
            i = 2
            while os.path.exists(dest):
                dest = os.path.join(audio_dir, f"{stem} {i}{ext}")
                i += 1
            os.replace(src, dest)
            os.rmdir(os.path.dirname(src))
            filepath = dest
    filepath_rel = None if filepath is None else os.path.relpath(filepath)
    transcribe_logger.info(json.dumps({"time": datetime.now().isoformat(), "input": filepath_rel, "output": text, "latency": f'{latency_ms} ms',
        "model": asr.model.name_or_path, "hardware": inference_hardware()}))

In [7]:
def transcribe_speech(filepath):
    latency_ms = None
    if filepath is None:
        gr.Warning("No audio found, please retry.")
        text = ""
    else:
        started = datetime.now()
        output = asr(filepath)
        latency_ms = round((datetime.now() - started).total_seconds() * 1000)
        text = output["text"]
    log_transcription(filepath, text, latency_ms)
    return text

In [8]:
def detect_intent(text):
    started = datetime.now()
    emb = intent_model.encode(text, normalize_embeddings=True)
    scores = util.cos_sim(emb, intent_emb)[0]
    probs = scores.softmax(dim=0)
    i = int(probs.argmax())
    latency_ms = round((datetime.now() - started).total_seconds() * 1000)
    return intents[i], round(float(probs[i]), 2), latency_ms

In [9]:
INTENT_LABELS = {
    "news": ["person", "organization", "place"],
    "ecommerce": ["item", "product"],
}

def extract_query(text, intent):
    spans = list(dict.fromkeys(
        e["text"] for e in ner_model.predict_entities(text, INTENT_LABELS[intent], threshold=0.4)
    ))
    if not spans:
        return "", 0.0
    emb = intent_model.encode([text, *spans], normalize_embeddings=True)
    target = intent_emb[intents.index(intent)]
    scores = util.cos_sim(emb[0], emb[1:])[0] + 2 * util.cos_sim(target, emb[1:])[0]
    probs = scores.softmax(dim=0)
    i = int(probs.argmax())
    return spans[i], round(float(probs[i]), 2)

In [10]:
cases = [
    ("Well tell me some news about Tom Cruise.", "news", "Tom Cruise"),
    ("What's up with Trump these days?", "news", "Trump"),
    ("Show me some t-shirts from Amazon.com", "ecommerce", "t-shirts"),
    ("get me some nyke shoes from amazon.com", "ecommerce", "nyke shoes"),
    ("find me some shoes similar to elon musk", "ecommerce", "shoes"),
    ("Tell me the latest headlines about Barack Obama.", "news", "Barack Obama"),
    ("What is happening in New York these days?", "news", "New York"),
    ("Any recent news about NASA?", "news", "NASA"),
    ("Find me wireless headphones from eBay.", "ecommerce", "wireless headphones"),
    ("Show me Adidas running shoes.", "ecommerce", "Adidas running shoes"),
    ("I want a jacket like the one Taylor Swift wore.", "ecommerce", "jacket"),
    ("What's the news about Apple's new iPhone?", "news", "Apple"),
    ("get me some addidas sneakers from flipkart", "ecommerce", "addidas sneakers"),
    ("Trump news", "news", "Trump"),
    ("looking for a cheap office chair", "ecommerce", "office chair"),
]

ok = 0
for text, actual_intent, actual_query in cases:
    intent = detect_intent(text)[0]
    query = extract_query(text, intent)[0]
    match = intent == actual_intent and query == actual_query
    ok += match
    print(text)
    print(f"{'ok' if match else 'WRONG'} ----- intent: {intent}, query: {query!r}")
    print()
print(f"ok: {ok}/{len(cases)}")

Batches: 100%|██████████| 1/1 [00:00<00:00, 256.19it/s]


Well tell me some news about Tom Cruise.
ok ----- intent: news, query: 'Tom Cruise'



Batches: 100%|██████████| 1/1 [00:00<00:00, 276.58it/s]


What's up with Trump these days?
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 302.53it/s]


...

get me some addidas sneakers from flipkart
ok ----- intent: ecommerce, query: 'addidas sneakers'



Batches: 100%|██████████| 1/1 [00:00<00:00, 352.85it/s]


Trump news
ok ----- intent: news, query: 'Trump'



Batches: 100%|██████████| 1/1 [00:00<00:00, 298.87it/s]

looking for a cheap office chair
ok ----- intent: ecommerce, query: 'office chair'

ok: 15/15


In [11]:
PDP_PATH = {
    "amazon.com": "inurl:/dp/",
    "ebay.com": "inurl:/itm/",
    "walmart.com": "inurl:/ip/",
    "target.com": "inurl:/p/",
    "bestbuy.com": "inurl:.p",
    "etsy.com": "inurl:/listing/",
    "flipkart.com": "inurl:/p/",
    "myntra.com": "inurl:/buy",
    "nike.com": "inurl:/t/",
}

def store_site(text):
    orgs = ner_model.predict_entities(text, ["organization"], threshold=0.4)
    people = {e["text"].lower() for e in ner_model.predict_entities(text, ["person"], threshold=0.3)}
    for org in orgs:
        if org["text"].lower() in people:
            continue
        name = org["text"].lower().removeprefix("www.").rstrip(".")
        return name if "." in name else f"{name}.com"
    return None

def product_query(text, query):
    site = store_site(text)
    if not site:
        return f"buy {query}"
    return f"buy {query} site:{site} {PDP_PATH.get(site, '')}".rstrip()

def news_hits(query):
    url = "https://news.google.com/rss/search?" + urllib.parse.urlencode({"q": query, "hl": "en-US", "gl": "US", "ceid": "US:en"})
    with urllib.request.urlopen(url, timeout=5) as response:
        root = ET.fromstring(response.read())
    return [{"title": item.findtext("title"), "url": item.findtext("link")} for item in root.findall("./channel/item")[:5]]

def product_hits(text, query):
    url = "https://api.search.brave.com/res/v1/web/search?" + urllib.parse.urlencode({"q": product_query(text, query), "count": 5})
    request = urllib.request.Request(url, headers={"Accept": "application/json", "X-Subscription-Token": os.environ.get("BRAVE_API_KEY")})
    with urllib.request.urlopen(request, timeout=5) as response:
        data = json.load(response)
    return [{"title": hit["title"], "url": hit["url"], "description": hit.get("description")} for hit in data.get("web", {}).get("results", [])[:5]]

def search_results(text):
    intent = detect_intent(text)[0]
    query = extract_query(text, intent)[0]
    hits = news_hits(query) if intent == "news" else product_hits(text, query)
    return intent, query, hits

In [12]:
search_results("Well tell me some news about Tom Cruise.")

Batches: 100%|██████████| 1/1 [00:00<00:00, 281.76it/s]


('news',
 'Tom Cruise',
 [{'title': "'Digger': Tom Cruise Movie Bombs at Box Office, Will Lose $150 Million - Deadline",
   'url': 'https://news.google.com/rss/articles/CBMiekFVX3lxTFBiUlBBbDhReUowdkJsS0JaeW9lN1RxX2dVbkk1VUdSWjI2VU1DUzUzU0dLVmZBWVB0ZUNBR1hYVEx1ekU1OHo3dEJ0NV9xdy04ekktbzlxSjZRSEJ0SldFdXB0MkE5Z3NlNGlBMFlXSC16Wk4tSW5sbUJB?oc=5'},
  {'title': 'Tom Cruise’s Digger appears to call out American critics who hated the movie after box office flop - yahoo.com',
   'url': 'https://news.google.com/rss/articles/CBMingFBVV95cUxOMGhvV2kwUzE0NEU2R0g5dFU2eXVLczdISXoxOFBaV19lNjN4bWNWQ25PcjBGQlB0c1F4NjJqT2ZuRUJzY1ZwYzMwb25pOUR3dXdtaDdkVVpkM1UxV2xPdkJ5VGtEUjZoMWluUXJqdDRDM1ZONDVvMU9fODhKSGtWVVl3dERVNEJEdVA2WlFsc21VdFNsaGlCdW5mU0JxUQ?oc=5'},
  {'title': 'Yes, Tom Cruise Should Be a Frontrunner for Best Actor - Awards Daily',
   'url': 'https://news.google.com/rss/articles/CBMimAFBVV95cUxQSDNoVTRWSmtuUUJwYWl4dXJIOV9zWTJCdlhaWmZrWC1jQUtoRlkyM0VVY0pQUF9wc0EtYk1XRkpLSDBtWUxIRHpoTjJYTXBXb2VhX0ti

In [13]:
search_results("Show me some t-shirts from Amazon.com")

Batches: 100%|██████████| 1/1 [00:00<00:00, 263.73it/s]


('ecommerce',
 't-shirts',
 [{'title': 'Amazon.com: Custom T-Shirt with Your Image for Men, Women and Kids by Modify by Amazon Merch on Demand : Clothing, Shoes & Jewelry',
   'url': 'https://www.amazon.com/Custom-T-Shirt-Your-Image-Green/dp/B09W4DK44V',
   'description': 'Amazon.com: Custom <strong>T</strong>-<strong>Shirt</strong> with Your Image for Men, Women and Kids by Modify by Amazon Merch on Demand : Clothing, Shoes &amp; Jewelry'},
  {'title': "BAIMORE Women's Fashionable Sexy Sheer Mesh Panel High-Neck Long-Sleeve Top for Night Outings and Office Wear Black at Amazon Women’s Clothing store",
   'url': 'https://www.amazon.com/BAIMORE-Fashionable-High-Neck-Long-Sleeve-Outings/dp/B0GJZLB5HG',
   'description': '<strong>Buy</strong> BAIMORE Women&#x27;s Fashionable Sexy Sheer Mesh Panel High-Neck Long-Sleeve Top for Night Outings and Office Wear Black and other <strong>T</strong>-<strong>Shirts</strong> at Amazon.com. Our wide selection is elegible for free shipping and free ret

In [ ]:
mic_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="microphone",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)
os.makedirs("logs/audio", exist_ok=True)
mic_transcribe.input_components[0].GRADIO_CACHE = os.path.abspath("logs/audio")

file_transcribe = gr.Interface(
    fn=transcribe_speech,
    inputs=gr.Audio(sources="upload",
                    type="filepath"),
    outputs=gr.Textbox(label="Transcription",
                       lines=3),
)

demo = gr.Blocks()

with demo:
    gr.TabbedInterface(
        [mic_transcribe,
         file_transcribe],
        ["Transcribe Microphone",
         "Transcribe Audio File"],
    )

demo.launch(share=True, 
            server_port=int(os.environ.get('PORT1', 7680)))

* Running on local URL:  http://127.0.0.1:7680
* Running on public URL: https://494c1feba99f511059.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


In [ ]:
demo.close()